<a href="https://colab.research.google.com/github/sibandze/Multimodedal-Bird-Intellegence-System/blob/main/notebooks/exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook: Data Loading + Spectogram Pipeline + Exploration

---



In [9]:
import os
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

In [7]:
DATA_DIR = '../data/'
AUDIO_DOWNLOADS_DIR = os.path.join(DATA_DIR, "audio_downloads")
os.makedirs(AUDIO_DOWNLOADS_DIR, exist_ok=True)
print(f"Audio download directory created at: {AUDIO_DOWNLOADS_DIR}")

NameError: name 'os' is not defined

In [11]:
csv_github_url = "https://raw.githubusercontent.com/sibandze/Multimodedal-Bird-Intellegence-System/main/data/birds_voices.csv"
df = pd.read_csv(csv_github_url)

df.head()

,common_name,scientific_name,recordist_name,recording_length,Date,TYPE,xc_id,Time,Country,Download_link
0,Common Ostrich,Struthio camelus australis,Frank Lambert,0:53,2019-10-30,call,XC516153,08:05,South Africa,https://xeno-canto.org/516153/download
1,Common Ostrich,Struthio camelus,Jeremy Hegge,0:26,2014-11-20,call,XC208209,04:00,South Africa,https://xeno-canto.org/208209/download
2,Common Ostrich,Struthio camelus,Jeremy Hegge,0:04,2014-11-21,call,XC208128,06:00,South Africa,https://xeno-canto.org/208128/download
3,Common Ostrich,Struthio camelus,Derek Solomon,0:11,2010-02-09,call,XC46725,07:00,South Africa,https://xeno-canto.org/46725/download
4,Common Ostrich,Struthio camelus,Morioka Zoological Park ZOOMO,1:47,2021-09-06,"voice during egg laying, zoo collection",XC675445,17:00,Japan,https://xeno-canto.org/675445/download


In [1]:
missing_scientific_names = df['scientific_name'].isnull().sum()
print(f"Number of missing scientific names: {missing_scientific_names}")

if missing_scientific_names == 0:
    print("All entries have scientific names.")
else:
    print(f"{missing_scientific_names} entries are missing scientific names. These entries should be addressed (e.g., dropped or filled) if scientific name is a required feature.")

NameError: name 'df' is not defined

In [2]:
# Create a mapping from scientific name to common name
scientific_to_common_name_map = df.set_index('scientific_name')['common_name'].drop_duplicates().to_dict()
print("Scientific Name to Common Name Map (first 5 entries):")
for i, (sci_name, comm_name) in enumerate(scientific_to_common_name_map.items()):
    if i >= 5: break
    print(f"  {sci_name}: {comm_name}")

# Get unique scientific names
unique_scientific_names = df['scientific_name'].unique()

# Create a dictionary to map scientific names to unique integer IDs
scientific_name_to_id = {name: i for i, name in enumerate(unique_scientific_names)}

print("\nScientific Name to ID Map (first 5 entries):")
for i, (sci_name, id_val) in enumerate(scientific_name_to_id.items()):
    if i >= 5: break
    print(f"  {sci_name}: {id_val}")

# Optionally, add the numerical ID to the DataFrame
df['scientific_name_id'] = df['scientific_name'].map(scientific_name_to_id)
display(df[['scientific_name', 'scientific_name_id']].head())

NameError: name 'df' is not defined

In [3]:
num_species = len(unique_scientific_names)
print(f"Number of different species: {num_species}")

NameError: name 'unique_scientific_names' is not defined

In [4]:
import requests

def download_audio(url, filename, output_dir="./audio_downloads"):
    """
    Downloads an audio file from a given URL.

    Args:
        url (str): The URL of the audio file.
        filename (str): The desired filename for the downloaded audio.
        output_dir (str): The directory to save the downloaded audio file.

    Returns:
        str: The full path to the downloaded file, or None if download fails.
    """
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() # Raise an exception for HTTP errors
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {filepath}")
        return filepath
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")
        return None


In [8]:
def download_audio(url, filename, output_dir=AUDIO_DOWNLOADS_DIR):
    """
    Downloads an audio file from a given URL.

    Args:
        url (str): The URL of the audio file.
        filename (str): The desired filename for the downloaded audio.
        output_dir (str): The directory to save the downloaded audio file.

    Returns:
        str: The full path to the downloaded file, or None if download fails.
    """
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() # Raise an exception for HTTP errors
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {filepath}")
        return filepath
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")
        return None

NameError: name 'AUDIO_DOWNLOADS_DIR' is not defined

In [5]:
def create_spectrogram(audio_path, sr=22050, n_fft=2048, hop_length=512):
    """
    Loads an audio file and generates its spectrogram.

    Args:
        audio_path (str): The path to the audio file.
        sr (int): Sampling rate for audio loading. Defaults to 22050.
        n_fft (int): FFT window size. Defaults to 2048.
        hop_length (int): Number of samples between successive frames. Defaults to 512.

    Returns:
        numpy.ndarray: The magnitude spectrogram.
    """
    try:
        # Load the audio file
        y, sr = librosa.load(audio_path, sr=sr)

        # Compute the Short-Time Fourier Transform (STFT)
        stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)

        # Convert to magnitude spectrogram
        spectrogram = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

        # Display the spectrogram
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(spectrogram, sr=sr, x_axis='time', y_axis='log', hop_length=hop_length)
        plt.colorbar(format='%+2.0f dB')
        plt.title('Spectrogram')
        plt.tight_layout()
        plt.show()

        return spectrogram
    except Exception as e:
        print(f"Error creating spectrogram for {audio_path}: {e}")
        return None


In [9]:
# Define directory for saving spectrograms
SPECTROGRAM_DIR = os.path.join(DATA_DIR, "spectrograms")
os.makedirs(SPECTROGRAM_DIR, exist_ok=True)
print(f"Spectrograms will be saved to: {SPECTROGRAM_DIR}")

# Modify create_spectrogram to return data and add a separate save function
def create_spectrogram_data(audio_path, sr=22050, n_fft=2048, hop_length=512):
    """
    Loads an audio file and generates its spectrogram data.

    Args:
        audio_path (str): The path to the audio file.
        sr (int): Sampling rate for audio loading. Defaults to 22050.
        n_fft (int): FFT window size. Defaults to 2048.
        hop_length (int): Number of samples between successive frames. Defaults to 512.

    Returns:
        tuple: (numpy.ndarray, int) The magnitude spectrogram data and sampling rate, or (None, None) if an error occurs.
    """
    try:
        y, sr_load = librosa.load(audio_path, sr=sr)
        stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
        spectrogram_data = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
        return spectrogram_data, sr_load
    except Exception as e:
        print(f"Error generating spectrogram data for {audio_path}: {e}")
        return None, None

def save_spectrogram_image(spectrogram_data, sr, output_path, hop_length=512):
    """
    Saves a spectrogram numpy array as an image.

    Args:
        spectrogram_data (numpy.ndarray): The magnitude spectrogram data.
        sr (int): Sampling rate.
        output_path (str): The full path to save the spectrogram image.
        hop_length (int): Number of samples between successive frames.
    """
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(spectrogram_data, sr=sr, x_axis='time', y_axis='log', hop_length=hop_length)
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram')
    plt.tight_layout()
    plt.savefig(output_path) # Save the figure
    plt.close() # Close the figure to prevent it from displaying in notebooks


NameError: name 'os' is not defined

Let's test these functions with an example. I will download the first audio file listed in your DataFrame and then generate its spectrogram.

In [6]:
# Get the first audio download link from the DataFrame
example_url = df['Download_link'].iloc[0]
example_filename = f"audio_{df['xc_id'].iloc[0]}.mp3" # Using xc_id for filename

# Download the audio file
downloaded_filepath = download_audio(example_url, example_filename)

# If download was successful, create and display the spectrogram
if downloaded_filepath:
    spectrogram = create_spectrogram(downloaded_filepath)
    if spectrogram is not None:
        print(f"Spectrogram generated for {example_filename}.")
    else:
        print(f"Failed to generate spectrogram for {example_filename}.")
else:
    print(f"Failed to download audio from {example_url}.")

NameError: name 'df' is not defined

Now, let's process all audio files in the DataFrame. We will download each audio, generate its spectrogram, save both locally, and update the DataFrame.

In [10]:
local_audio_paths = []
local_spectrogram_paths = []

for index, row in df.iterrows():
    audio_url = row['Download_link']
    xc_id = row['xc_id']
    scientific_name = row['scientific_name']

    # Generate filenames based on xc_id and scientific_name for uniqueness and clarity
    audio_filename = f"{xc_id}_{scientific_name.replace(' ', '_').replace('/', '_')}.mp3"
    spectrogram_filename = f"{xc_id}_{scientific_name.replace(' ', '_').replace('/', '_')}.png"

    downloaded_audio_path = download_audio(audio_url, audio_filename, output_dir=AUDIO_DOWNLOADS_DIR)

    if downloaded_audio_path:
        spectrogram_data, sr_loaded = create_spectrogram_data(downloaded_audio_path)
        if spectrogram_data is not None:
            spectrogram_output_path = os.path.join(SPECTROGRAM_DIR, spectrogram_filename)
            save_spectrogram_image(spectrogram_data, sr_loaded, spectrogram_output_path)
            local_audio_paths.append(downloaded_audio_path)
            local_spectrogram_paths.append(spectrogram_output_path)
            print(f"Processed {audio_filename}")
        else:
            local_audio_paths.append(downloaded_audio_path)
            local_spectrogram_paths.append(None) # Mark as failed if spectrogram not created
            print(f"Failed to create spectrogram for {audio_filename}")
    else:
        local_audio_paths.append(None) # Mark as failed if download failed
        local_spectrogram_paths.append(None)
        print(f"Failed to download audio from {audio_url}")

# Add the new paths to the DataFrame
df['local_audio_path'] = local_audio_paths
df['local_spectrogram_path'] = local_spectrogram_paths

# Display a sample of the updated DataFrame
display(df[['common_name', 'scientific_name', 'Download_link', 'local_audio_path', 'local_spectrogram_path']].head())

# Save the updated DataFrame to a new CSV file
updated_csv_path = os.path.join(DATA_DIR, 'birds_voices_with_local_paths.csv')
df.to_csv(updated_csv_path, index=False)
print(f"\nUpdated DataFrame saved to: {updated_csv_path}")

NameError: name 'df' is not defined

---

### Data Exploration and Class Imbalance Analysis

As requested, we will now focus on exploring the data, checking for class imbalance, and simplifying the DataFrame to only include `common_name`, `scientific_name`, and `Download_link`.

In [11]:
# Reload the original CSV to start fresh with a trimmed DataFrame
csv_github_url = "https://raw.githubusercontent.com/sibandze/Multimodedal-Bird-Intellegence-System/main/data/birds_voices.csv"
df = pd.read_csv(csv_github_url)

# Keep only the columns of interest
df = df[['common_name', 'scientific_name', 'Download_link']].copy()

# Re-create the scientific_name_id mapping for classification
unique_scientific_names = df['scientific_name'].unique()
scientific_name_to_id = {name: i for i, name in enumerate(unique_scientific_names)}
df['scientific_name_id'] = df['scientific_name'].map(scientific_name_to_id)

print("DataFrame after trimming columns and re-creating scientific_name_id:")
display(df.head())
print(f"Number of entries in the DataFrame: {len(df)}")

NameError: name 'pd' is not defined

Now, let's analyze the class imbalance by counting the occurrences of each scientific name and visualizing the distribution.

In [12]:
species_counts = df['scientific_name'].value_counts()

print("Top 10 most frequent species:\n", species_counts.head(10))
print("\nBottom 10 least frequent species:\n", species_counts.tail(10))

# Visualize the distribution of species
plt.figure(figsize=(15, 6))
species_counts.plot(kind='bar')
plt.title('Distribution of Scientific Names (Species)')
plt.xlabel('Scientific Name')
plt.ylabel('Number of Recordings')
plt.xticks(rotation=90, fontsize=8) # Rotate x-axis labels for better readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

NameError: name 'df' is not defined

The bar chart above clearly illustrates the distribution of recordings across different species. We can observe significant class imbalance, with some species having a very high number of recordings while many others have only a few. This imbalance is common in real-world datasets and will need to be considered during model training (e.g., through techniques like stratification, oversampling, or undersampling).